## BTreeMap Null Separated Keys
---

In [2]:
use std::iter;
use std::collections::BTreeMap;
use std::collections::HashMap;

In [3]:
#[derive(Debug)]
struct PasswordEntry {
    title: String,
    username: String,
    password: String,
}

In [4]:
let mut nskmap = BTreeMap::<String, String>::new();
nskmap.insert("password_entry\x00bank\x00username".into(), "myname".into());
nskmap.insert("password_entry\x00bank\x00password".into(), "opensesame".into());
nskmap.insert("password_entry\x00email\x00username".into(), "myname".into());
nskmap.insert("password_entry\x00email\x00password".into(), "123456".into());
nskmap

{"password_entry\0bank\0password": "opensesame", "password_entry\0bank\0username": "myname", "password_entry\0email\0password": "123456", "password_entry\0email\0username": "myname"}

In [5]:
fn fetch_tuples(nskmap: &BTreeMap<String, String>, type_key: &str, key: &str) -> Vec<(String, String)> {
    nskmap.range(format!("{}\x00{}\x00", type_key, key)..format!("{}\x00{}\x01", type_key, key)).map(
        |(k,v)| (k.split("\x00").nth(2).unwrap_or("").to_string(), v.clone())
    )
    .chain(iter::once(("title".to_string(), key.to_string())))
    .collect::<Vec<(String, String)>>()
}

In [6]:
fetch_tuples(&nskmap, "password_entry", "bank")

[("password", "opensesame"), ("username", "myname"), ("title", "bank")]

In [7]:
impl FromIterator<(String, String)> for PasswordEntry {
    fn from_iter<I: IntoIterator<Item = (String, String)>>(iter: I) -> Self {
        let mut map: HashMap<_, _> = iter.into_iter().collect();
        Self {
            title: map.remove("title").unwrap_or_default(),
            username: map.remove("username").unwrap_or_default(),
            password: map.remove("password").unwrap_or_default(),
        }
    }
}

In [8]:
fn fetch(nskmap: &BTreeMap<String, String>, type_key: &str, key: &str) -> PasswordEntry {
    fetch_tuples(nskmap, type_key, key).into_iter().collect()
}

In [9]:
fetch(&nskmap, "password_entry", "bank")

PasswordEntry { title: "bank", username: "myname", password: "opensesame" }

In [10]:
fetch(&nskmap, "password_entry", "email")

PasswordEntry { title: "email", username: "myname", password: "123456" }

In [11]:
fn into_nskmap(nskmap: &mut BTreeMap<String, String>, entry: &PasswordEntry) {
    nskmap.insert(format!("password_entry\x00{}\x00username", entry.title), entry.username.clone());
    nskmap.insert(format!("password_entry\x00{}\x00password", entry.title), entry.password.clone());
}

In [12]:
let mut nskmap2 = BTreeMap::<String, String>::new();
into_nskmap(&mut nskmap2, &PasswordEntry { title: "bank".into(), username: "myname".into(), password: "opensesame".into() });
into_nskmap(&mut nskmap2, &PasswordEntry { title: "email".into(), username: "myname".into(), password: "123456".into() });
nskmap2

{"password_entry\0bank\0password": "opensesame", "password_entry\0bank\0username": "myname", "password_entry\0email\0password": "123456", "password_entry\0email\0username": "myname"}